# Gemma-2-9b-it Fine-tuning on Healthcare Dataset

Run this code file in Kaggle

Turn on GPU P100

Set API keys in secret folder for running step 5

## Step 1: Install Dependencies

In [ ]:
%%capture
# Install all required packages
!pip install -U transformers>=4.42.3
!pip install -U datasets
!pip install -U accelerate
!pip install -U peft
!pip install -U trl
!pip install -U bitsandbytes
!pip install -U wandb
!pip install -U scikit-learn

print("All packages installed!")


## Step 2: Imports & Setup

In [ ]:
import os
import gc
import json
import warnings
from dataclasses import dataclass
from datetime import datetime

import numpy as np
import torch
import transformers
import peft as peft_lib

from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
)
from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from trl import SFTTrainer, SFTConfig
import bitsandbytes as bnb

# Suppress noisy warnings
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# version checks 
print(f" Start time : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f" PyTorch    : {torch.__version__}")
print(f" Transformers: {transformers.__version__}")
print(f" PEFT       : {peft_lib.__version__}")
print(f" GPU        : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f" VRAM       : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

## Step 3: Configuration (QLoRA + Training)

In [ ]:
@dataclass
class Config:
    # ── Model ────────────────────────────────────────────────────────────────
    base_model: str       = "google/gemma-2-9b-it"
    new_model: str        = "gemma-2-9b-it-healthcare"
    dataset_name: str     = "lavita/ChatDoctor-HealthCareMagic-100k"

    # ── Data ─────────────────────────────────────────────────────────────────
    # Use 5000 samples for quick test; change to None for full dataset
    num_samples: int      = 5000
    test_size: float      = 0.1        # 10% for validation
    max_seq_length: int   = 512        # P100 friendly

    # ── QLoRA ────────────────────────────────────────────────────────────────
    lora_r: int           = 16
    lora_alpha: int       = 32         # 2 × lora_r (standard)
    lora_dropout: float   = 0.05
    lora_bias: str        = "none"
    # Freeze first N layers (of 42 total) — from Notebook 1
    freeze_layers: int    = 16

    # ── Training ─────────────────────────────────────────────────────────────
    num_train_epochs: int = 1
    per_device_train_batch_size: int = 1   # Keep low for P100 16GB
    per_device_eval_batch_size: int  = 1
    gradient_accumulation_steps: int = 4   # Effective batch = 4
    learning_rate: float  = 2e-4
    warmup_steps: int     = 20
    optim: str            = "paged_adamw_32bit"
    lr_scheduler_type: str = "cosine"
    logging_steps: int    = 10
    eval_steps: float     = 0.2        # Evaluate 5 times per epoch
    save_steps: int       = 100
    fp16: bool            = True       # P100 supports fp16
    bf16: bool            = False      # P100 does NOT support bf16
    group_by_length: bool = True       # Speed up training
    packing: bool         = False


cfg = Config()

# ── Auto-detect dtype ────────────────────────────────────────────────────────
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    torch_dtype          = torch.bfloat16
    cfg.fp16             = False
    cfg.bf16             = True
    attn_implementation  = "flash_attention_2"
    print("Using bfloat16 + FlashAttention2")
else:
    torch_dtype          = torch.float16
    attn_implementation  = "eager"
    print("Using float16 (P100 mode) + eager attention")

print("\n Configuration:")
print(f"   Model       : {cfg.base_model}")
print(f"   Samples     : {cfg.num_samples}")
print(f"   LoRA r      : {cfg.lora_r}  |  alpha: {cfg.lora_alpha}")
print(f"   Freeze layers: {cfg.freeze_layers} / 42")
print(f"   Batch size  : {cfg.per_device_train_batch_size} × {cfg.gradient_accumulation_steps} accum = {cfg.per_device_train_batch_size * cfg.gradient_accumulation_steps} effective")
print(f"   Max seq len : {cfg.max_seq_length}")

## Step 4: Authentication (HuggingFace + W&B)

In [ ]:
# HuggingFace login (needed for gated models like Gemma)
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    import wandb

    secrets = UserSecretsClient()
    hf_token = secrets.get_secret("HUGGINGFACE_TOKEN")
    login(token=hf_token)
    print("HuggingFace login successful")

    wb_token = secrets.get_secret("wandb")
    wandb.login(key=wb_token)
    run = wandb.init(
        project="Gemma-2-9b-Healthcare-QLoRA",
        job_type="training",
        anonymous="allow",
    )
    print("W&B login successful")
    REPORT_TO = "wandb"

except Exception as e:
    print(f" Auth skipped (running locally or secrets not set): {e}")
    REPORT_TO = "none"

## Step 5: Load Model & Tokenizer (QLoRA Setup)

In [ ]:
# QLoRA: 4-bit quantization config (from Notebook 1 + Notebook 3)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # NF4 quantization (QLoRA paper)
    bnb_4bit_compute_dtype=torch_dtype, # Compute in fp16/bf16
    bnb_4bit_use_double_quant=True,     # Double quantization saves ~0.4 bits/param
)

print("Loading model in 4-bit QLoRA mode...")
model = AutoModelForCausalLM.from_pretrained(
    cfg.base_model,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch_dtype,
    attn_implementation=attn_implementation,
)
model.config.use_cache = False          # Disable KV cache during training
model.config.pretraining_tp = 1        # Tensor parallelism = 1

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    cfg.base_model,
    trust_remote_code=True,
)
tokenizer.padding_side = "right"       # Pad on right (required for causal LM)
tokenizer.add_eos_token = True         # Append <eos> at end (from Notebook 1)

# Fix: Gemma tokenizer may not have a pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded! Parameters: {model.num_parameters() / 1e9:.2f}B")
print(f" Vocab size  : {tokenizer.vocab_size}")
print(f" Pad token   : {tokenizer.pad_token}")

## Step 6: LoRA Configuration (Selective Layer Targeting)

In [ ]:
# Helper: find all linear modules (from Notebook 3)
def find_all_linear_names(model):
    """Find all 4-bit linear layers to apply LoRA adapters."""
    target_cls = bnb.nn.Linear4bit
    module_names = set()
    for name, module in model.named_modules():
        if isinstance(module, target_cls):
            parts = name.split('.')
            module_names.add(parts[0] if len(parts) == 1 else parts[-1])
    module_names.discard('lm_head')
    return sorted(list(module_names))


target_modules = find_all_linear_names(model)
print(f"🎯 Target LoRA modules ({len(target_modules)}): {target_modules}")

# Selective layer targeting (freeze first 16 layers)
total_layers = 42
layers_to_transform = [i for i in range(total_layers) if i >= cfg.freeze_layers]
print(f"\n🧊 Frozen layers    : 0 – {cfg.freeze_layers - 1}")
print(f"🔥 Trainable layers : {cfg.freeze_layers} – {total_layers - 1} ({len(layers_to_transform)} layers)")

# LoRA Config
lora_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    target_modules=target_modules,
    layers_to_transform=layers_to_transform,
    lora_dropout=cfg.lora_dropout,
    bias=cfg.lora_bias,
    task_type="CAUSAL_LM",
)

# ⚠️ Sirf kbit training ke liye prepare karo
# get_peft_model() mat karo — SFTTrainer khud peft_config se LoRA apply karega
model = prepare_model_for_kbit_training(model)

print("✅ Model prepared for kbit training!")
print("   LoRA will be applied by SFTTrainer via peft_config")

## Step 7: Load & Prepare Dataset

In [ ]:
print(f" Loading dataset: {cfg.dataset_name}")
print(f"   Using {cfg.num_samples} samples (set cfg.num_samples=None for full dataset)")

# Load dataset — use select(range(N)) to avoid OOM (from project spec)
raw_dataset = load_dataset(cfg.dataset_name, split="all")
raw_dataset = raw_dataset.shuffle(seed=42).select(range(cfg.num_samples))

print(f" Dataset loaded: {len(raw_dataset)} samples")
print(f"   Columns: {raw_dataset.column_names}")
print(f"\n Sample entry:")
sample = raw_dataset[0]
print(f"   instruction: {sample['instruction'][:80]}...")
print(f"   input      : {sample['input'][:80]}...")
print(f"   output     : {sample['output'][:80]}...")

# ── Chat template formatting 
# IMPORTANT: Gemma-2 does NOT support 'system' role in chat template.
# Fix: merge instruction into the 'user' message directly.

def format_chat_template(row):
    """
    Gemma-2 only accepts 'user' and 'assistant' roles.
    System instruction is prepended into the user turn.

      user      → instruction + '\n\n' + patient query
      assistant → doctor response
    """
    # Merge system instruction + patient input into single user message
    user_content = f"{row['instruction']}\n\n{row['input']}"

    messages = [
        {"role": "user",      "content": user_content},
        {"role": "assistant", "content": row["output"]},
    ]
    row["text"] = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return row


print("🔄 Formatting dataset with chat template...")
print("   ℹ️  Gemma-2 has no system role → merging instruction into user turn")
dataset = raw_dataset.map(
    format_chat_template,
    num_proc=1,           # num_proc=1 — avoids multiprocess pickling errors
    remove_columns=[c for c in raw_dataset.column_names if c != "text"],
    desc="Formatting",
)

# ── Train/validation split ──────────────────────────────────────────────────
dataset = dataset.train_test_split(test_size=cfg.test_size, seed=42)
train_ds = dataset["train"]
eval_ds  = dataset["test"]

print(f"\n Dataset ready!")
print(f"   Train : {len(train_ds)} samples")
print(f"   Valid : {len(eval_ds)} samples")

print(f"\n Formatted sample (first 500 chars):")
print(train_ds[0]["text"][:500])

## Step 8: Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir=cfg.new_model,

    # Epochs & Steps 
    num_train_epochs=cfg.num_train_epochs,

    # Batch & Gradient 
    per_device_train_batch_size=cfg.per_device_train_batch_size,
    per_device_eval_batch_size=cfg.per_device_eval_batch_size,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    gradient_checkpointing=True,

    # Optimizer & LR 
    optim=cfg.optim,
    learning_rate=cfg.learning_rate,
    weight_decay=0.001,
    warmup_steps=cfg.warmup_steps,
    lr_scheduler_type=cfg.lr_scheduler_type,

    # Precision 
    fp16=cfg.fp16,
    bf16=cfg.bf16,

    # Logging & Eval
    logging_steps=cfg.logging_steps,
    eval_strategy="steps",
    eval_steps=100,        # ← absolute number (same type as save_steps)
    save_strategy="steps",
    save_steps=100,        # ← dono equal hain, load_best_model_at_end works
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    # Misc 
    report_to=REPORT_TO,
    dataloader_pin_memory=False,
)

print("Training arguments set!")
print(f"   Effective batch size : {cfg.per_device_train_batch_size * cfg.gradient_accumulation_steps}")
print(f"   Optimizer            : {cfg.optim}")
print(f"   Learning rate        : {cfg.learning_rate}")
print(f"   Precision            : {'bf16' if cfg.bf16 else 'fp16'}")

## Step 9: SFT Trainer Setup & Training

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir=cfg.new_model,

    # ── Epochs & Steps ───────────────────────────────────────────────────────
    num_train_epochs=cfg.num_train_epochs,

    # ── Batch & Gradient ─────────────────────────────────────────────────────
    per_device_train_batch_size=cfg.per_device_train_batch_size,
    per_device_eval_batch_size=cfg.per_device_eval_batch_size,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_grad_norm=0.0,   # ← FIX: bf16/fp16 scaler conflict — clip karna band

    # ── Optimizer & LR ───────────────────────────────────────────────────────
    optim=cfg.optim,
    learning_rate=cfg.learning_rate,
    weight_decay=0.001,
    warmup_steps=cfg.warmup_steps,
    lr_scheduler_type=cfg.lr_scheduler_type,

    # ── Precision ────────────────────────────────────────────────────────────
    fp16=False,
    bf16=False,

    # ── Logging & Eval ───────────────────────────────────────────────────────
    logging_steps=cfg.logging_steps,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    # ── Misc ─────────────────────────────────────────────────────────────────
    report_to=REPORT_TO,
    dataloader_pin_memory=False,

    # ── SFT-specific ─────────────────────────────────────────────────────────
    dataset_text_field="text",
    packing=cfg.packing,
    max_length=cfg.max_seq_length,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    peft_config=lora_config,
    processing_class=tokenizer,
)

print("✅ SFTTrainer ready!")
print(f"   Train batches : {len(trainer.get_train_dataloader())}")
print(f"   Eval batches  : {len(trainer.get_eval_dataloader())}")

# Show GPU memory before training
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved()  / 1e9
    total     = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"💾 GPU Memory — Allocated: {allocated:.2f} GB | Reserved: {reserved:.2f} GB | Total: {total:.2f} GB")

print("\n Starting training...\n")
train_result = trainer.train()

print("\n✅ Training complete!")
print(f"   Steps        : {train_result.global_step}")
print(f"   Training loss: {train_result.training_loss:.4f}")
print(f"   Runtime      : {train_result.metrics['train_runtime'] / 60:.1f} min")

## Step 10: Save Fine-tuned Model

In [ ]:
# Re-enable cache for inference
model.config.use_cache = True

# Save LoRA adapter weights locally
trainer.model.save_pretrained(cfg.new_model)
tokenizer.save_pretrained(cfg.new_model)

print(f" Model adapter saved to: ./{cfg.new_model}/")

# Finish W&B run
try:
    wandb.finish()
    print("W&B run finished")
except:
    pass

# Optional: Push to HuggingFace Hub 
# Uncomment to push adapter to your HF Hub repo:
# trainer.model.push_to_hub(cfg.new_model, use_temp_dir=False)
# print(f" Model pushed to HuggingFace Hub: {cfg.new_model}")

## Step 11: Inference & Testing
Testing the fine-tuned model — inspired by Notebook 2's approach of extracting the last-token logit from the causal LM head for classification/generation.

In [ ]:
def generate_response(
    model,
    tokenizer,
    patient_query: str,
    system_prompt: str = "If you are a doctor, please answer the medical questions based on the patient's description.",
    max_new_tokens: int = 200,
) -> str:
    """
    Generate a doctor response given a patient query.
    Gemma-2 does NOT support system role — system_prompt is
    prepended into the user message (must match training format).
    """
    # Merge system prompt + query into user turn (same as training format)
    user_content = f"{system_prompt}\n\n{patient_query}"

    messages = [
        {"role": "user", "content": user_content},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,    # Appends <start_of_turn>model\n
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=cfg.max_seq_length,
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only newly generated tokens (skip input prompt)
    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    response   = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return response.strip()


print("✅ Inference function ready!")

In [ ]:
# Test with 3 healthcare queries
test_queries = [
    "Hello, I am having a severe anxiety and panic attack. Could you help me?",
    "I have been experiencing chest pain for the past 2 days, especially when I breathe deeply. I am 35 years old.",
    "What are the symptoms of Type 2 Diabetes and how can it be managed through diet?",
]

model.eval()
print("=" * 70)
print(" Healthcare Q&A — Fine-tuned Gemma-2-9b-it")
print("=" * 70)

for i, query in enumerate(test_queries, 1):
    print(f"\n Query {i}:")
    print(f"   {query}")
    print(f"\n Response:")
    response = generate_response(model, tokenizer, query)
    print(f"   {response}")
    print("-" * 70)

## Step 12: Load Saved Model & Re-test (Unsloth-style Reload)
This section demonstrates how to reload the saved LoRA adapter and run inference.

In [ ]:
# Free memory before reloading 
print("  Freeing GPU memory...")
del model
del trainer
gc.collect()
torch.cuda.empty_cache()

allocated = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
print(f" Memory freed. Current GPU usage: {allocated:.2f} GB")

In [ ]:
# Reload base model + merge LoRA adapter (Notebook 2 style) 
print(" Reloading base model in 4-bit...")

bnb_config_infer = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_use_double_quant=True,
)

base_model_reload = AutoModelForCausalLM.from_pretrained(
    cfg.base_model,
    quantization_config=bnb_config_infer,
    device_map="auto",
    torch_dtype=torch_dtype,
    attn_implementation=attn_implementation,
)

# Load LoRA adapter on top
inference_model = PeftModel.from_pretrained(base_model_reload, cfg.new_model)
inference_model = inference_model.eval()

# Reload tokenizer
inference_tokenizer = AutoTokenizer.from_pretrained(cfg.new_model)
if inference_tokenizer.pad_token is None:
    inference_tokenizer.pad_token = inference_tokenizer.eos_token

print(" Model reloaded with LoRA adapter!")

In [ ]:
# Re-run inference with reloaded model 
query = "I have been having headaches every morning for the past week. I also feel dizzy sometimes. What could be the cause?"

print("=" * 70)
print(" Testing Reloaded Model")
print("=" * 70)
print(f"\n Patient Query:\n   {query}")

response = generate_response(inference_model, inference_tokenizer, query)
print(f"\n Doctor Response:\n   {response}")

## Step 13: Training Summary

In [ ]:
print("=" * 70)
print(" TRAINING SUMMARY")
print("=" * 70)
print(f"""
  Model: {cfg.base_model}
  Fine-tuned on: {cfg.dataset_name}
  Samples used: {cfg.num_samples} (train: {int(cfg.num_samples * 0.9)}, val: {int(cfg.num_samples * 0.1)})

  Technique: QLoRA (4-bit NF4 quantization + LoRA adapters)
  LoRA rank (r): {cfg.lora_r}  |  alpha: {cfg.lora_alpha}
  Frozen layers: 0 – {cfg.freeze_layers - 1}  (trainable: {cfg.freeze_layers}–41)
  Target modules: All 4-bit linear layers

  Batch size: {cfg.per_device_train_batch_size} × {cfg.gradient_accumulation_steps} grad accum = {cfg.per_device_train_batch_size * cfg.gradient_accumulation_steps} effective
  Max seq length: {cfg.max_seq_length}
  Optimizer: {cfg.optim}
  Learning rate: {cfg.learning_rate}
  Precision: {'bf16' if cfg.bf16 else 'fp16'}

  Saved to: ./{cfg.new_model}/
""")

print("Fine-tuning complete! Model is ready for healthcare Q&A.")

## Step 14: Zip or Folder (download model weights)

In [ ]:
import shutil
import os

# Zip banao — sirf adapter files (base model nahi, woh 5GB+ hai)
print("Creating zip of fine-tuned LoRA adapter...")

# Jo files chahiye next project ke liye:
# adapter_model.safetensors — LoRA weights
# adapter_config.json       — LoRA config
# tokenizer.json            — tokenizer
# tokenizer_config.json     — tokenizer config
# chat_template.jinja       — Gemma-2 chat template
# special_tokens_map.json   — special tokens

zip_path = shutil.make_archive(
    base_name="/kaggle/working/gemma-2-9b-it-healthcare-adapter",
    format="zip",
    root_dir="/kaggle/working",
    base_dir="gemma-2-9b-it-healthcare",
)

size_mb = os.path.getsize(zip_path) / 1e6
print(f"Zip created: {zip_path}")
print(f"   Size: {size_mb:.1f} MB")
print(f"\n Download from Kaggle sidebar:")
print(f"   Output → gemma-2-9b-it-healthcare-adapter.zip")

In [ ]:
import shutil
import os

src = "/kaggle/working/gemma-2-9b-it-healthcare"
dst = "/kaggle/working/Ali Sajid"

# Create destination folder
os.makedirs(dst, exist_ok=True)

# Files to copy
files_to_copy = [
    "adapter_model.safetensors",
    "adapter_config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "chat_template.jinja",
    "special_tokens_map.json",
    "README.md",
]

print(" Copying files to 'Ali Sajid' folder...\n")

for fname in files_to_copy:
    src_file = os.path.join(src, fname)
    dst_file = os.path.join(dst, fname)
    if os.path.exists(src_file):
        shutil.copy2(src_file, dst_file)
        size_mb = os.path.getsize(dst_file) / 1e6
        print(f"    {fname}  ({size_mb:.1f} MB)")
    else:
        print(f"     {fname}  — not found, skipping")

print(f"\n Done! Folder ready: /kaggle/working/Ali Sajid/")
print(f"\n Kaggle Output tab mein jaao:")
print(f"   Output → Ali Sajid → files one by one download karo")

# List final folder contents
print(f"\n Files in folder:")
for f in os.listdir(dst):
    size = os.path.getsize(os.path.join(dst, f)) / 1e6
    print(f"   {f}  ({size:.1f} MB)")

In [ ]:
from huggingface_hub import hf_hub_download
import shutil

# HuggingFace se download karo
path = hf_hub_download(
    repo_id="google/gemma-2-9b-it",
    filename="special_tokens_map.json",
)

# Ali Sajid folder mein copy karo
shutil.copy2(path, "/kaggle/working/Ali Sajid/special_tokens_map.json")
print(" special_tokens_map.json downloaded and copied!")